# 01 · DataFrames and research file formats

**Module:** 2 · **Time:** 40 minutes · **Prerequisites:** Module 1 fundamentals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/01_dataframes_intro.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F01_dataframes_intro.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** The repository is currently private and these badges refer to `main` after merge.

## Learning goals

- **I**dentify rows, columns, and data types in a DataFrame.
- **R**ead synthetic data from CSV, Excel, Parquet, and Stata formats.
- **I**nspect a research dataset before analysis.

In [ ]:
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print(survey.head(2))

## Why this matters

Survey datasets arrive in files, not as Python lists. A **DataFrame** organizes observations into rows and variables into columns. Before calculating statistics, inspect dimensions, column names, types, and missing values. This lesson uses a six-row **fictional teaching extract** whose field names resemble the larger survey generator; it is not a random sample or evidence about real households.

## Explore a DataFrame

Use `head`, `shape`, and `dtypes` before attempting analysis. The index is a row label, not necessarily a unique respondent identifier.

In [ ]:
print("Rows and columns:", survey.shape)
print("Column names:", survey.columns.tolist())
print(survey.dtypes)
print(survey.head(3))

## Round-trip a CSV

A CSV stores text, not full pandas types. We use a memory buffer so learners need no downloaded files, and pandas infers types upon reading.

In [ ]:
from io import StringIO

csv_text = survey.to_csv(index=False)
csv_data = pd.read_csv(StringIO(csv_text))
print("CSV rows:", len(csv_data))
print(csv_data.head(2))

## Compare Excel and Parquet

Excel is common for handoffs; Parquet stores column types efficiently. Temporary files are removed automatically after the example.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    base = Path(directory)
    survey.to_excel(base / "survey.xlsx", index=False)
    excel_rows = pd.read_excel(base / "survey.xlsx")
    if find_spec("pyarrow") is not None:
        survey.to_parquet(base / "survey.parquet", index=False)
        parquet_rows = pd.read_parquet(base / "survey.parquet")
        print("Parquet rows:", len(parquet_rows))
    else:
        print("Parquet requires pyarrow; run uv sync --locked.")
print("Excel rows:", len(excel_rows))

## Stata labels as categorical values

For categorical employment status, pandas can write Stata value labels and recover them as a categorical column when reading `.dta` files. This is a synthetic practice file.

In [ ]:
labeled = survey[["person_id", "employment"]].copy()
labeled["employment"] = labeled["employment"].astype("category")
with TemporaryDirectory() as directory:
    stata_path = Path(directory) / "survey.dta"
    labeled.to_stata(stata_path, write_index=False, version=118)
    stata_data = pd.read_stata(stata_path, convert_categoricals=True)
print(stata_data["employment"].dtype)
print(stata_data.head(2))

## If you know Stata/R

| Task | Python / pandas | Stata | R |
| --- | --- | --- | --- |
| Read CSV | `pd.read_csv(path)` | `import delimited using ...` | `readr::read_csv(path)` |
| Read Excel | `pd.read_excel(path)` | `import excel using ...` | `readxl::read_excel(path)` |
| Read Stata | `pd.read_stata(path)` | `use file.dta` | `haven::read_dta(path)` |
| Preview | `survey.head()` | `list in 1/5` | `head(survey)` |

For SPSS and SAS, see the `pyreadstat` package after its installation and data permissions are verified. Index labels are not guaranteed to identify people.

## Try it

Complete each blank answer cell, then run its friendly check. In the learner version, checks print hints instead of raising an exception.

### ★ Exercise 1 — Count survey records

Store the number of DataFrame rows in `n_records`.

In [ ]:
n_records = None

In [ ]:
if isinstance(n_records, int) and n_records == len(survey):
    print("✓ Record count correct")
else:
    print("Hint: len(survey) gives the number of rows.")

### ★ Exercise 2 — Select numeric weights

Store the `survey_weight` column (a Series) in `weights`.

In [ ]:
weights = None

In [ ]:
expected_weights = survey["survey_weight"]
if isinstance(weights, pd.Series) and weights.equals(expected_weights):
    print("✓ Survey weights selected")
else:
    print("Hint: put the column name inside square brackets.")

### ★★ Exercise 3 — Inspect a research file

Create `column_names` as a Python list containing every variable name in the survey.

In [ ]:
column_names = None

In [ ]:
if isinstance(column_names, list) and column_names == survey.columns.tolist():
    print("✓ Column names are correct")
else:
    print("Hint: DataFrame.columns has a tolist() method.")

### ★★★ Exercise 4 — Read back the CSV

Set `csv_roundtrip` to a DataFrame made by reading the CSV text in `csv_text`.

In [ ]:
csv_roundtrip = None

In [ ]:
expected_csv = pd.read_csv(StringIO(csv_text))
if isinstance(csv_roundtrip, pd.DataFrame) and csv_roundtrip.equals(expected_csv):
    print("✓ CSV round trip completed")
else:
    print("Hint: wrap csv_text in StringIO, then call pd.read_csv(...).")

## Common errors

- `FileNotFoundError` — the working directory or filename is wrong. This lesson avoids that problem by using temporary files.
- Unexpected data types — a CSV cannot preserve every original dtype; inspect `dtypes` after importing.
- Duplicate IDs — the DataFrame row index does not guarantee that `person_id` is unique.
- Missing optional engine — Excel and Parquet readers require dependencies such as `openpyxl` and `pyarrow`, included in the course's locked environment.

## Recap

- A DataFrame has rows, named columns, and an index.
- Inspect shape, types, and identifiers before analysis.
- CSV, Excel, Stata, and Parquet carry different type information.
- Temporary practice files and synthetic data protect reproducibility.

## Next steps

Next: [Selecting and filtering](02_selecting_filtering.ipynb). Read [pandas' introductory tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) and include the lesson path.